# Snowflake-Native Geocoding Demo

End-to-end geocoding using **Overture Maps** Marketplace share — no external APIs needed.

This notebook demonstrates:
1. **UDF Deployment** — `PARSE_ADDRESS` (ML-based address parser) and `STANDARDIZE_STREET` (USPS abbreviation expander)
2. **Reverse Geocode** — Lat/Lon → nearest address(es) via `ST_DWITHIN`
3. **Forward Geocode** — Free-text address → coordinates via parse + standardize + JOIN

### Prerequisites
- Overture Maps address share installed: `DS_OVERTURE_MAPS__ADDRESSES.CARTO.ADDRESS`
- A warehouse with sufficient compute (MEDIUM recommended for batch operations)
- Database/schema for UDF deployment (default: `VENKAT_DB.PUBLIC`)

## 0. Initialization

Set execution context. No connection code needed — Workspace provides the session automatically.

In [ ]:
%%sql -r init_context
USE DATABASE VENKAT_DB;
USE SCHEMA PUBLIC;

In [ ]:
# Configuration — shared database and table references
OVERTURE_TABLE = "DS_OVERTURE_MAPS__ADDRESSES.CARTO.ADDRESS"
TARGET_DB = "VENKAT_DB"
TARGET_SCHEMA = "PUBLIC"

print(f"Overture table: {OVERTURE_TABLE}")
print(f"UDF target: {TARGET_DB}.{TARGET_SCHEMA}")

---
## 1. Deploy PARSE_ADDRESS UDF

A Python UDF that tokenizes free-text US addresses into structured fields using the
[usaddress](https://github.com/datamade/usaddress) library (CRF-based ML model).

**Input:** `'125 Constitution Dr, Menlo Park, CA 94025'`  
**Output:** `{"number": "125", "street": "Constitution Dr", "city": "Menlo Park", "state": "CA", "zip": "94025", "parse_error": false}`

The UDF pulls `usaddress` from Snowflake's shared PyPI repository — no staging required.

In [ ]:
%%sql -r create_parse_address
CREATE OR REPLACE FUNCTION PARSE_ADDRESS(raw_address VARCHAR)
  RETURNS OBJECT
  LANGUAGE PYTHON
  RUNTIME_VERSION = 3.11
  ARTIFACT_REPOSITORY = snowflake.snowpark.pypi_shared_repository
  PACKAGES = ('usaddress')
  HANDLER = 'parse'
AS $$
import usaddress

def parse(raw_address):
    if not raw_address:
        return None
    try:
        tagged, addr_type = usaddress.tag(raw_address)
    except usaddress.RepeatedLabelError:
        return {
            "number": None,
            "street": raw_address,
            "city": None,
            "state": None,
            "zip": None,
            "parse_error": True
        }

    street_parts = []
    for key in ['StreetNamePreDirectional', 'StreetName', 'StreetNamePostType', 'StreetNamePostDirectional']:
        if key in tagged:
            street_parts.append(tagged[key])

    return {
        "number": tagged.get('AddressNumber'),
        "street": ' '.join(street_parts) if street_parts else None,
        "city": tagged.get('PlaceName'),
        "state": tagged.get('StateName'),
        "zip": tagged.get('ZipCode'),
        "parse_error": False
    }
$$;

### Test PARSE_ADDRESS

Verify the UDF handles various address formats correctly.

In [ ]:
%%sql -r parse_test
SELECT 
    addr,
    PARSE_ADDRESS(addr):number::STRING AS number,
    PARSE_ADDRESS(addr):street::STRING AS street,
    PARSE_ADDRESS(addr):city::STRING AS city,
    PARSE_ADDRESS(addr):state::STRING AS state,
    PARSE_ADDRESS(addr):zip::STRING AS zip
FROM (SELECT * FROM VALUES 
    ('125 Constitution Dr, Menlo Park, CA 94025'),
    ('1600 Pennsylvania Ave NW, Washington, DC 20500'),
    ('350 5th Ave, New York, NY 10118'),
    ('8601 W Cross Dr, Littleton, CO 80123')
    AS t(addr)
);

---
## 2. Deploy STANDARDIZE_STREET UDF

A Python UDF that expands USPS-abbreviated street names to full Overture Maps format.

Real-world addresses use abbreviations (`E 38TH ST`, `JACK RABBIT TRL`, `W ANDERSON LN`),
but Overture stores full names (`East 38th Street`, `Jack Rabbit Trail`, `West Anderson Lane`).
This UDF bridges that gap with 230+ rules from USPS Publication 28 and Nominatim.

**Transformations:**
- Directional prefixes: `E` → `East`, `NW` → `Northwest`
- Street type suffixes: `ST` → `Street`, `TRL` → `Trail`, `BLVD` → `Boulevard`
- Multi-word abbreviations: `CR` → `County Road`, `FM` → `Farm to Market Road`
- Directional suffixes: `MAIN ST N` → `Main Street North`
- Title Case normalization

In [ ]:
%%sql -r create_standardize_street
CREATE OR REPLACE FUNCTION STANDARDIZE_STREET(street VARCHAR)
  RETURNS VARCHAR
  LANGUAGE PYTHON
  RUNTIME_VERSION = 3.11
  HANDLER = 'standardize'
AS $$
import re

# Directional expansions (used for both prefix and suffix positions)
DIRECTIONALS = {
    'N': 'North', 'S': 'South', 'E': 'East', 'W': 'West',
    'NE': 'Northeast', 'NW': 'Northwest', 'SE': 'Southeast', 'SW': 'Southwest',
    'STH': 'South',  # Australian/informal but seen in some US data
}

# Multi-word abbreviations: checked FIRST (before splitting into single words)
# These are prefix patterns that replace the first 1-2 words
MULTI_WORD_ABBREVS = {
    'CR': 'County Road',
    'CO RD': 'County Road',
    'COUNTY RD': 'County Road',
    'SR': 'State Route',
    'SH': 'State Highway',
    'STATE HWY': 'State Highway',
    'STATE RD': 'State Road',
    'US HWY': 'US Highway',
    'US RT': 'US Route',
    'US RTE': 'US Route',
    'FM': 'Farm to Market Road',
    'RM': 'Ranch to Market Road',
    'RR': 'Ranch Road',
    'FSR': 'Forest Service Road',
    'IH': 'Interstate Highway',
}

# Street type suffix expansions — comprehensive dictionary sourced from:
#   1. USPS Publication 28 Appendix C (official USPS abbreviations)
#   2. Nominatim variants-en.yaml (community-maintained OSM abbreviations)
#
# Format: ABBREVIATION → Full Word (as Overture stores it)
# Multiple abbreviations can map to the same full word.
STREET_TYPES = {
    # --- A ---
    'AL': 'Alley', 'ALY': 'Alley', 'ALLY': 'Alley',
    'ANX': 'Anex',
    'ARC': 'Arcade',
    'AV': 'Avenue', 'AVE': 'Avenue', 'AVN': 'Avenue',
    # --- B ---
    'BCH': 'Beach',
    'BND': 'Bend',
    'BLF': 'Bluff',
    'BLFS': 'Bluffs',
    'BLVD': 'Boulevard', 'BVD': 'Boulevard',
    'BR': 'Branch',
    'BRG': 'Bridge', 'BRDG': 'Bridge', 'BDGE': 'Bridge',
    'BRK': 'Brook',
    'BRKS': 'Brooks',
    'BYP': 'Bypass', 'BYPA': 'Bypass', 'BPS': 'Bypass',
    'BWAY': 'Broadway', 'BDWY': 'Broadway', 'BWY': 'Broadway',
    # --- C ---
    'CP': 'Camp',
    'CYN': 'Canyon',
    'CPE': 'Cape',
    'CSWY': 'Causeway', 'CAUS': 'Causeway',
    'CTR': 'Center', 'CEN': 'Center', 'CNTR': 'Center',
    'CTRS': 'Centers',
    'CIR': 'Circle',
    'CIRS': 'Circles',
    'CCT': 'Circuit',
    'CLF': 'Cliff',
    'CLFS': 'Cliffs',
    'CL': 'Close',
    'CLB': 'Club',
    'CMN': 'Common', 'COMM': 'Common',
    'CMNS': 'Commons',
    'COR': 'Corner', 'CNR': 'Corner', 'CRN': 'Corner',
    'CORS': 'Corners',
    'CRSE': 'Course',
    'CT': 'Court', 'CRT': 'Court',
    'CTS': 'Courts',
    'CV': 'Cove', 'COV': 'Cove',
    'CVS': 'Coves',
    'CK': 'Creek', 'CRK': 'Creek',  # CR omitted — handled as "County Road" in MULTI_WORD_ABBREVS
    'CRES': 'Crescent',
    'CRST': 'Crest', 'CST': 'Crest',
    'XING': 'Crossing', 'CRSG': 'Crossing',
    'XRD': 'Crossroad',
    'XRDS': 'Crossroads',
    'CURV': 'Curve', 'CVE': 'Curve',
    # --- D ---
    'DL': 'Dale', 'DLE': 'Dale',
    'DM': 'Dam',
    'DV': 'Divide',
    'DR': 'Drive', 'DRV': 'Drive',
    'DRS': 'Drives',
    'DRWY': 'Driveway', 'DVWY': 'Driveway',
    # --- E ---
    'EST': 'Estate',
    'ESTS': 'Estates',
    'EXP': 'Expressway', 'EXPY': 'Expressway', 'EXPWY': 'Expressway', 'XWAY': 'Expressway',
    'EXT': 'Extension', 'EX': 'Extension',
    'EXTS': 'Extensions',
    # --- F ---
    'FLS': 'Falls',
    'FRY': 'Ferry', 'FY': 'Ferry',
    'FLD': 'Field', 'FD': 'Field',
    'FLDS': 'Fields',
    'FLT': 'Flat', 'FL': 'Flat',
    'FLTS': 'Flats',
    'FRD': 'Ford',
    'FRDS': 'Fords',
    'FRST': 'Forest',
    'FRG': 'Forge',
    'FRGS': 'Forges',
    'FRK': 'Fork',
    'FRKS': 'Forks',
    'FT': 'Fort',
    'FWY': 'Freeway', 'FRWY': 'Freeway',
    # --- G ---
    'GDN': 'Garden',
    'GDNS': 'Gardens',
    'GTWY': 'Gateway', 'GWY': 'Gateway',
    'GL': 'Glade', 'GLD': 'Glade', 'GLDE': 'Glade',
    'GLN': 'Glen',
    'GLNS': 'Glens',
    'GRN': 'Green',
    'GRNS': 'Greens',
    'GRV': 'Grove', 'GR': 'Grove', 'GRO': 'Grove',
    'GRVS': 'Groves',
    # --- H ---
    'HBR': 'Harbor',
    'HBRS': 'Harbors',
    'HVN': 'Haven',
    'HTS': 'Heights', 'HGTS': 'Heights', 'HT': 'Heights',
    'HWY': 'Highway',
    'HL': 'Hill',
    'HLS': 'Hills',
    'HOLW': 'Hollow',
    # --- I ---
    'INLT': 'Inlet',
    'IS': 'Island',
    'ISS': 'Islands',
    # --- J ---
    'JCT': 'Junction', 'JCTN': 'Junction', 'JNC': 'Junction',
    'JCTS': 'Junctions',
    # --- K ---
    'KY': 'Key',
    'KYS': 'Keys',
    'KNL': 'Knoll',
    'KNLS': 'Knolls',
    # --- L ---
    'LK': 'Lake',
    'LKS': 'Lakes',
    'LDG': 'Landing', 'LNDG': 'Landing',
    'LN': 'Lane', 'LA': 'Lane',
    'LGT': 'Light',
    'LGTS': 'Lights',
    'LF': 'Loaf',
    'LCK': 'Lock',
    'LCKS': 'Locks',
    'LP': 'Loop',
    # --- M ---
    'ML': 'Mall',
    'MNR': 'Manor',
    'MNRS': 'Manors',
    'MDW': 'Meadow',
    'MDWS': 'Meadows',
    'MWS': 'Mews',
    'MSN': 'Mission',
    'MTWY': 'Motorway',
    'MT': 'Mount',
    'MTN': 'Mountain',
    'MTNS': 'Mountains',
    # --- N ---
    'NCK': 'Neck',
    # --- O ---
    'ORCH': 'Orchard',
    'OPAS': 'Overpass',
    # --- P ---
    'PK': 'Park',
    'PKS': 'Parks',
    'PKWY': 'Parkway', 'PKY': 'Parkway', 'PWY': 'Parkway',
    'PASS': 'Pass',
    'PSGE': 'Passage',
    'PATH': 'Path',
    'PHWY': 'Pathway', 'PWAY': 'Pathway',
    'PNE': 'Pine',
    'PNES': 'Pines',
    'PL': 'Place',
    'PLN': 'Plain',
    'PLNS': 'Plains',
    'PLZ': 'Plaza', 'PLZA': 'Plaza',
    'PNT': 'Point', 'PT': 'Point',
    'PTS': 'Points',
    'PRT': 'Port',
    'PRTS': 'Ports',
    'PR': 'Prairie',
    # --- Q ---
    'QY': 'Quay',
    # --- R ---
    'RADL': 'Radial',
    'RNCH': 'Ranch',
    'RGE': 'Range', 'RNGE': 'Range',
    'RPD': 'Rapid',
    'RPDS': 'Rapids',
    'RST': 'Rest',
    'RDG': 'Ridge', 'RDGE': 'Ridge',
    'RDGS': 'Ridges',
    'RIV': 'River', 'RVR': 'River',
    'RD': 'Road',
    'RDS': 'Roads',
    'RTE': 'Route', 'RT': 'Route',
    'ROW': 'Row',
    'RUN': 'Run',
    # --- S ---
    'SHL': 'Shoal',
    'SHLS': 'Shoals',
    'SHR': 'Shore',
    'SHRS': 'Shores',
    'SKWY': 'Skyway',
    'SPG': 'Spring',
    'SPGS': 'Springs',
    'SPUR': 'Spur',
    'SQ': 'Square',
    'SQS': 'Squares',
    'STA': 'Station', 'STN': 'Station',
    'STRA': 'Stravenue',
    'STRM': 'Stream',
    'ST': 'Street',
    'STS': 'Streets',
    'SMT': 'Summit',
    # --- T ---
    'TCE': 'Terrace', 'TER': 'Terrace', 'TERR': 'Terrace',
    'THFR': 'Thoroughfare',
    'TRCE': 'Trace',
    'TRAK': 'Track', 'TRK': 'Track', 'TR': 'Track',
    'TRFY': 'Trafficway',
    'TRL': 'Trail',
    'TRLR': 'Trailer',
    'TUN': 'Tunnel', 'TUNL': 'Tunnel',
    'TPK': 'Turnpike', 'TPKE': 'Turnpike',
    # --- U ---
    'UPAS': 'Underpass',
    'UN': 'Union',
    'UNS': 'Unions',
    # --- V ---
    'VLY': 'Valley', 'VY': 'Valley',
    'VLYS': 'Valleys',
    'VIA': 'Viaduct', 'VDCT': 'Viaduct', 'VIAD': 'Viaduct',
    'VW': 'View',
    'VWS': 'Views',
    'VLG': 'Village', 'VILL': 'Village',
    'VLGS': 'Villages',
    'VL': 'Ville',
    'VIS': 'Vista', 'VST': 'Vista', 'VSTA': 'Vista',
    # --- W ---
    'WALK': 'Walk', 'WK': 'Walk', 'WLK': 'Walk',
    'WALL': 'Wall',
    'WY': 'Way',
    'WL': 'Well',
    'WLS': 'Wells',
    # --- Service road / direction markers (stripped) ---
    'SVRD': '', 'NB': '', 'SB': '', 'WB': '', 'EB': '',
}

# Directionals that can appear as suffix (last word after street type is handled)
DIRECTIONAL_SUFFIXES = {
    'N': 'North', 'S': 'South', 'E': 'East', 'W': 'West',
    'NE': 'Northeast', 'NW': 'Northwest', 'SE': 'Southeast', 'SW': 'Southwest',
}


def standardize(street):
    if not street:
        return None

    s = street.strip().upper()
    if not s:
        return None

    # --- Step 1: Multi-word abbreviation expansion (prefix match) ---
    # Check longest matches first to avoid partial matches
    for abbrev in sorted(MULTI_WORD_ABBREVS.keys(), key=len, reverse=True):
        if s == abbrev or s.startswith(abbrev + ' '):
            remainder = s[len(abbrev):].strip()
            s = MULTI_WORD_ABBREVS[abbrev] + (' ' + remainder if remainder else '')
            break

    words = s.split()
    if not words:
        return None

    # --- Step 2: Expand directional prefix (first word only) ---
    if words[0] in DIRECTIONALS:
        words[0] = DIRECTIONALS[words[0]]

    # --- Step 3: Strip trailing service road markers ---
    while len(words) > 1 and words[-1] in ('SVRD', 'NB', 'SB', 'WB', 'EB'):
        words.pop()

    # --- Step 4: Expand directional suffix (last word, if it's a bare directional) ---
    # e.g., "MAIN ST N" → after suffix expansion: "MAIN ST North"
    # Must check BEFORE street type expansion since directional is after the type
    if len(words) > 2 and words[-1] in DIRECTIONAL_SUFFIXES:
        # Only treat as directional suffix if second-to-last is a street type
        if words[-2] in STREET_TYPES:
            words[-1] = DIRECTIONAL_SUFFIXES[words[-1]]

    # --- Step 5: Expand street type suffix (last word, or second-to-last if last is directional) ---
    if words:
        # Check if last word is now an expanded directional (from step 4)
        # In that case, the street type is second-to-last
        expanded_dirs = set(DIRECTIONAL_SUFFIXES.values())
        if len(words) > 2 and words[-1] in expanded_dirs:
            target_idx = -2
        else:
            target_idx = -1

        target_word = words[target_idx]
        if target_word in STREET_TYPES:
            expanded = STREET_TYPES[target_word]
            if expanded:
                words[target_idx] = expanded
            else:
                words.pop(target_idx)  # strip empty expansions

    # --- Step 6: Title case the result ---
    result = ' '.join(words)
    return result.title()
$$;


### Test STANDARDIZE_STREET

Verify abbreviation expansion and title-casing.

In [ ]:
%%sql -r std_test
SELECT input, STANDARDIZE_STREET(input) AS standardized
FROM (SELECT * FROM VALUES
    ('E 38TH ST'),
    ('JACK RABBIT TRL'),
    ('W ANDERSON LN'),
    ('CR 250'),
    ('N MAIN ST'),
    ('CONGRESS AVE'),
    ('FM 1325'),
    ('S LAMAR BLVD')
    AS t(input)
);

---
## 3. Reverse Geocode (Lat/Lon → Address)

Find the nearest addresses to a given point using `ST_DWITHIN` spatial filter.

**How it works:**
1. `ST_DWITHIN(geometry, point, radius_meters)` filters addresses within the radius
2. `ST_DISTANCE` computes exact distance for ranking
3. Results ordered by proximity

This example searches for addresses within 200m of a point in Salt Lake City, UT.

In [ ]:
%%sql -r reverse_single
-- Reverse Geocode: Find nearest addresses to a lat/lon point
-- Example: Salt Lake City (40.760681, -111.890913)
SELECT
    a.ID,
    a.NUMBER,
    a.STREET,
    a.UNIT,
    a.POSTCODE,
    a.POSTAL_CITY,
    ST_Y(a.GEOMETRY) AS RESULT_LAT,
    ST_X(a.GEOMETRY) AS RESULT_LON,
    ROUND(ST_DISTANCE(a.GEOMETRY, TO_GEOGRAPHY('POINT(-111.890913 40.760681)')), 1) AS DISTANCE_M
FROM DS_OVERTURE_MAPS__ADDRESSES.CARTO.ADDRESS a
WHERE ST_DWITHIN(a.GEOMETRY, TO_GEOGRAPHY('POINT(-111.890913 40.760681)'), 200)
  AND a.COUNTRY = 'US'
ORDER BY DISTANCE_M
LIMIT 5;

### Batch Reverse Geocode

Reverse geocode multiple points at once using a JOIN + deduplication.
Each input point gets its single closest address.

In [ ]:
%%sql -r reverse_batch
-- Batch reverse geocode: multiple points → nearest address for each
WITH input_points AS (
    SELECT * FROM VALUES
        ('SLC_Office',    40.760681, -111.890913),
        ('Austin_Capitol', 30.274670, -97.740349),
        ('NYC_ESB',       40.748817, -73.985428)
    AS t(label, lat, lon)
),
ranked AS (
    SELECT
        p.label,
        p.lat AS input_lat,
        p.lon AS input_lon,
        a.NUMBER || ' ' || a.STREET AS full_address,
        a.POSTAL_CITY,
        a.POSTCODE,
        ST_Y(a.GEOMETRY) AS result_lat,
        ST_X(a.GEOMETRY) AS result_lon,
        ROUND(ST_DISTANCE(a.GEOMETRY, TO_GEOGRAPHY('POINT(' || p.lon || ' ' || p.lat || ')')), 1) AS distance_m,
        ROW_NUMBER() OVER (
            PARTITION BY p.label
            ORDER BY ST_DISTANCE(a.GEOMETRY, TO_GEOGRAPHY('POINT(' || p.lon || ' ' || p.lat || ')'))
        ) AS rn
    FROM input_points p
    JOIN DS_OVERTURE_MAPS__ADDRESSES.CARTO.ADDRESS a
      ON ST_DWITHIN(a.GEOMETRY, TO_GEOGRAPHY('POINT(' || p.lon || ' ' || p.lat || ')'), 200)
     AND a.COUNTRY = 'US'
)
SELECT label, input_lat, input_lon, full_address, POSTAL_CITY, POSTCODE, result_lat, result_lon, distance_m
FROM ranked
WHERE rn = 1
ORDER BY label;

---
## 4. Forward Geocode (Address → Lat/Lon)

Convert free-text addresses to coordinates using the full pipeline:

```
Raw Address → PARSE_ADDRESS → STANDARDIZE_STREET → JOIN to Overture → Ranked Results
```

**Join strategy:**
- Filter by `COUNTRY = 'US'` and `NUMBER` (exact match on house number)
- Match on `POSTCODE` OR `POSTAL_CITY` (fallback since 38% of Overture US has NULL city)
- `LIKE '%standardized_street%'` for street name containment
- Rank by `EDITDISTANCE` (Levenshtein) for best match

### Single Address Forward Geocode

Parse and geocode a single address in one query using a CTE.

In [ ]:
%%sql -r forward_single
-- Forward geocode a single free-text address
WITH parsed AS (
    SELECT
        '175 E 400 S, Salt Lake City, UT 84111' AS raw_address,
        PARSE_ADDRESS('175 E 400 S, Salt Lake City, UT 84111') AS addr
),
fields AS (
    SELECT
        raw_address,
        addr:number::STRING AS number,
        addr:street::STRING AS street,
        STANDARDIZE_STREET(addr:street::STRING) AS street_std,
        addr:city::STRING AS city,
        addr:zip::STRING AS zip
    FROM parsed
)
SELECT
    f.raw_address,
    f.street_std AS input_street_standardized,
    a.NUMBER AS result_number,
    a.STREET AS result_street,
    a.POSTAL_CITY AS result_city,
    a.POSTCODE AS result_zip,
    ST_Y(a.GEOMETRY) AS result_lat,
    ST_X(a.GEOMETRY) AS result_lon,
    EDITDISTANCE(UPPER(a.STREET), UPPER(f.street_std)) AS edit_dist
FROM fields f
JOIN DS_OVERTURE_MAPS__ADDRESSES.CARTO.ADDRESS a
  ON a.COUNTRY = 'US'
 AND a.NUMBER = f.number
 AND (a.POSTCODE = f.zip OR UPPER(a.POSTAL_CITY) = UPPER(f.city))
 AND UPPER(a.STREET) LIKE '%' || UPPER(f.street_std) || '%'
ORDER BY edit_dist
LIMIT 5;

### Batch Forward Geocode

Forward geocode multiple addresses at once. Each address gets its top-ranked match
based on `EDITDISTANCE` (street similarity).

In [ ]:
%%sql -r forward_batch
-- Batch forward geocode: parse + standardize + match multiple addresses
WITH input_addresses AS (
    SELECT * FROM VALUES
        ('175 E 400 S, Salt Lake City, UT 84111'),
        ('1600 Pennsylvania Ave NW, Washington, DC 20500'),
        ('8601 W Cross Dr, Littleton, CO 80123'),
        ('350 5th Ave, New York, NY 10118'),
        ('100 Congress Ave, Austin, TX 78701')
    AS t(raw_address)
),
parsed AS (
    SELECT
        raw_address,
        PARSE_ADDRESS(raw_address):number::STRING AS number,
        PARSE_ADDRESS(raw_address):street::STRING AS street,
        PARSE_ADDRESS(raw_address):city::STRING AS city,
        PARSE_ADDRESS(raw_address):zip::STRING AS zip
    FROM input_addresses
),
geocoded AS (
    SELECT
        p.raw_address,
        STANDARDIZE_STREET(p.street) AS input_street_std,
        a.NUMBER AS result_number,
        a.STREET AS result_street,
        a.POSTAL_CITY AS result_city,
        a.POSTCODE AS result_zip,
        ST_Y(a.GEOMETRY) AS result_lat,
        ST_X(a.GEOMETRY) AS result_lon,
        EDITDISTANCE(UPPER(a.STREET), UPPER(STANDARDIZE_STREET(p.street))) AS edit_dist,
        ROW_NUMBER() OVER (
            PARTITION BY p.raw_address
            ORDER BY EDITDISTANCE(UPPER(a.STREET), UPPER(STANDARDIZE_STREET(p.street)))
        ) AS result_rank
    FROM parsed p
    JOIN DS_OVERTURE_MAPS__ADDRESSES.CARTO.ADDRESS a
      ON a.COUNTRY = 'US'
     AND a.NUMBER = p.number
     AND (a.POSTCODE = p.zip OR UPPER(a.POSTAL_CITY) = UPPER(p.city))
     AND UPPER(a.STREET) LIKE '%' || UPPER(STANDARDIZE_STREET(p.street)) || '%'
    WHERE p.number IS NOT NULL
      AND p.street IS NOT NULL
      AND (p.zip IS NOT NULL OR p.city IS NOT NULL)
)
SELECT raw_address, input_street_std, result_number, result_street, result_city, result_zip,
       result_lat, result_lon, edit_dist
FROM geocoded
WHERE result_rank = 1
ORDER BY raw_address;

---
## Summary

This notebook demonstrated Snowflake-native geocoding using Overture Maps data:

| Operation | Approach | Key Functions |
|-----------|----------|---------------|
| Address Parsing | ML tokenizer (usaddress) | `PARSE_ADDRESS()` |
| Street Normalization | 230+ USPS/Nominatim rules | `STANDARDIZE_STREET()` |
| Reverse Geocode | Spatial proximity search | `ST_DWITHIN()`, `ST_DISTANCE()` |
| Forward Geocode | Parse + Normalize + JOIN | All of the above + `EDITDISTANCE()` |

**Performance notes:**
- Reverse geocode: sub-second for single points, scales linearly for batch
- Forward geocode: ~30s for 100K addresses on MEDIUM warehouse (97.8% hit rate on Austin permits)
- First call to `PARSE_ADDRESS` may be slow (~30s) due to PyPI package install; subsequent calls are fast